In [ ]:
# install elan + Lean 4 toolchain
!curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | bash -s -- -y --default-toolchain leanprover/lean4:v4.11.0
import os
os.environ['PATH'] = '/root/.elan/bin:' + os.environ['PATH']
!lean --version

In [ ]:
import os, json, subprocess
from pathlib import Path

WORK = Path('/kaggle/working/verify')
WORK.mkdir(exist_ok=True)

# locate harvested.jsonl from harvest-v2 kernel_sources mount
HARVEST = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'harvested.jsonl' in files:
        HARVEST = os.path.join(root, 'harvested.jsonl')
        break
assert HARVEST, 'harvested.jsonl not found'
print('HARVEST:', HARVEST)

# load first row with proofs
with open(HARVEST) as f:
    rows = [json.loads(l) for l in f if l.strip()]
print(f'rows: {len(rows)}')
sample = rows[0]
print(f'sample id={sample["id"]} eq1={sample["eq1"]} eq2={sample["eq2"]} label={sample["label"]}')
print(f'proofs: {len(sample["proofs"])}')
for i, p in enumerate(sample['proofs']):
    print(f'  [{i}]: {p[:150]}')

In [ ]:
# build minimal Lean file with Magma class + theorem + proof, check via lean cli
import os, subprocess, time, re, textwrap

PREAMBLE = '''class Magma (G : Type) where
  op : G \u2192 G \u2192 G
infixl:70 " \u25c7 " => Magma.op
'''

def to_diamond(s):
    return s.replace('*', '\u25c7')

def used_vars(eqs, candidates='xyzwu'):
    return ''.join(v for v in candidates if re.search(rf'\b{v}\b', eqs))

def reindent(body):
    # strip 'by\n' prefix, dedent, then prepend 'by\n' and re-indent 2 spaces
    s = body.strip()
    if s.startswith('by'):
        s = s[2:].lstrip('\n')
    s = textwrap.dedent(s)
    indented = '\n'.join('  ' + ln if ln.strip() else '' for ln in s.split('\n'))
    return 'by\n' + indented

def build_lean(eq1, eq2, proof_body):
    eq1d, eq2d = to_diamond(eq1), to_diamond(eq2)
    vars_used = used_vars(eq1d + ' ' + eq2d)
    if not vars_used:
        vars_used = 'x'
    vlist = ' '.join(vars_used)
    proof = reindent(proof_body)
    return (PREAMBLE +
        'theorem sair_implication\n'
        '    (G : Type) [inst : Magma G]\n'
        f'    (h : \u2200 {vlist} : G, {eq1d})\n'
        f'    : \u2200 {vlist} : G, {eq2d} := ' + proof + '\n')

def check_proof(lean_src, timeout=60):
    p = WORK / 'test.lean'
    p.write_text(lean_src)
    t0 = time.time()
    try:
        r = subprocess.run(['lean', str(p)], capture_output=True, text=True, timeout=timeout)
        return {'ok': r.returncode == 0, 'rc': r.returncode,
                'stdout': r.stdout[:500], 'stderr': r.stderr[:500],
                'elapsed': time.time() - t0}
    except subprocess.TimeoutExpired:
        return {'ok': False, 'rc': -1, 'stdout': '', 'stderr': 'TIMEOUT', 'elapsed': time.time()-t0}

# test only label=true rows (false ones have no proof by definition)
true_rows = [r for r in rows if r.get('label') == True][:8]
print(f'testing {len(true_rows)} true-labeled rows')
results = []
for ri, row in enumerate(true_rows):
    for pi, proof in enumerate(row['proofs']):
        src = build_lean(row['eq1'], row['eq2'], proof)
        res = check_proof(src)
        results.append((ri, pi, res))
        tag = 'PASS' if res['ok'] else 'fail'
        print(f'row{ri} proof{pi} {tag} rc={res["rc"]} {res["elapsed"]:.2f}s')
        if not res['ok']:
            print('  err:', res['stdout'][:200].replace(chr(10), ' | ')[:200])

passed = sum(1 for _, _, r in results if r['ok'])
print(f'\n--- SMOKE TOTAL: {passed}/{len(results)} compiled ---')

# show one sample lean src for debugging
print('\n--- sample built lean src (row 0 proof 0) ---')
print(build_lean(rows[0]['eq1'], rows[0]['eq2'], rows[0]['proofs'][0]))